# Advection-Diffusion (Multichannel) Simulator

This notebook runs the `AdvectionDiffusionMultichannel` simulator — a 2D incompressible fluid solver driven by a vorticity initial condition.

- **State variables**: `vorticity`, `u` (x-velocity), `v` (y-velocity), `streamfunction`.
- **Conditioning variables**: `nu` (kinematic viscosity), `mu` (advection / forcing strength).
- **Dynamics**: spectral Poisson solve (FFT) for streamfunction; central finite differences for spatial derivatives; `scipy` RK45 time integration.
- **Boundary conditions**: periodic in both spatial directions.

### Why this is useful

Exposing all four fluid channels simultaneously makes this a natural benchmark for multi-target PDE surrogate models. The rich vorticity dynamics across a range of viscosities and forcing strengths produce diverse trajectories — from laminar diffusion to turbulent-like swirl patterns.


## Governing equations

The vorticity–streamfunction formulation of the 2D incompressible Navier-Stokes equations:

$$
\frac{\partial \omega}{\partial t} + (\mathbf{u} \cdot \nabla)\omega = \nu \nabla^2 \omega + \mu f(\omega),
$$

with velocity recovered from the streamfunction $\psi$ via

$$
\nabla^2 \psi = -\omega,
\qquad
u = \frac{\partial \psi}{\partial y},\ v = -\frac{\partial \psi}{\partial x}.
$$

### Boundary conditions

Periodic in both spatial directions on the domain $[0, L]^2$:

$$
\omega(0, y, t) = \omega(L, y, t), \qquad \omega(x, 0, t) = \omega(x, L, t).
$$

### Parameters

- **`nu`**: kinematic viscosity. Smaller values produce more active, longer-lived vortex structures.
- **`mu`**: advection / forcing strength controlling the intensity of nonlinear transport.


In [ ]:
from IPython.display import HTML

from autosim.simulations import AdvectionDiffusionMultichannel
from autosim.utils import plot_spatiotemporal_video

In [ ]:
sim = AdvectionDiffusionMultichannel(
    return_timeseries=True,
    log_level="progress_bar",
    n=64,
    L=10.0,
    T=80.25,
    dt=0.25,
    output_indices=[0],  # vorticity
    parameters_range={
        "nu": (0.0001, 0.01),
        "mu": (0.5, 2.0),
    },
)

batch = sim.forward_samples_spatiotemporal(n=2, random_seed=42)

print(
    "data shape:",
    batch["data"].shape,
    "[batch, time, x, y, channels={vorticity}]",
)
print("constant_scalars shape:", batch["constant_scalars"].shape)
print("sampled params (nu, mu):", batch["constant_scalars"])

In [ ]:
anim = plot_spatiotemporal_video(
    batch["data"],
    batch_idx=0,
    channel_names=["vorticity"],
    preserve_aspect=True,
)
HTML(anim.to_jshtml())